In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict
from items import Item
import matplotlib.pyplot as plt

load_dotenv()

hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

%matplotlib inline


In [ ]:
dataset = load_dataset("McAuley-lab/Amazon-Reviews-2023", f"raw_meta_Appliances", split="full", trust_remote_code=True)

print(f"Number of Appliances: {len(dataset):,}")

In [ ]:
datapoint = dataset[0]

In [ ]:
print(datapoint["title"])
print(datapoint["description"])
print(datapoint["features"])
print(datapoint["details"])
print(datapoint["price"])

In [ ]:
prices = 0

for datapoint in dataset:
    try:
        price = float(datapoint["price"])
        if price > 0:
            prices += 1
    except ValueError as e:
        pass

print(f"There are {prices:,} datapoints with a valid price.")

In [ ]:
prices = []
lengths = []

for datapoint in dataset:
    try:
        price = float(datapoint["price"])
        if price > 0:
            prices.append(price)
            contents = datapoint["title"] + str(datapoint["description"]) + str(datapoint["features"]) + str(datapoint["details"])
            lengths.append(len(contents))
    except ValueError as e:
        pass

In [ ]:
# Plot the distribution of lengths
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 6))
plt.title(f"Lengths: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel('Length (chars)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="lightblue", bins=range(0, 6000, 100))
plt.show()

In [ ]:
# Plot the distribution of prices

plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 1000, 10))
plt.show()

In [ ]:
for datapoint in dataset:
    try:
        price = float(datapoint["price"])
        if price > 21000:
            print(f"{datapoint['title']}: ${price:,.2f}")
    except ValueError as e:
        pass

In [ ]:
items = []

for datapoint in dataset:
    try:
        price = float(datapoint["price"])
        if price > 0:
            item = Item(datapoint, price)
            if item.include:
                items.append(item)
    except ValueError as e:
        pass


print(f"There are {len(items):,} items with a valid price and length.")


In [ ]:
items[0]

In [ ]:
print(items[100].prompt)

In [ ]:
print(items[100].test_prompt())

In [ ]:
tokens = [item.token_count for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Tokens: Avg {sum(tokens)/len(tokens):,.0f} and highest {max(tokens):,}\n")
plt.hist(tokens, rwidth=0.7, color="green", bins=range(0, 300, 10))

In [ ]:
prices = [item.price for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="purple", bins=range(0, 300, 10))
plt.show()